# 04.1.1 - Supervised Learning: Ekstraksi Aturan Bisnis dengan Decision Tree

## 1. Pengertian Decision Tree
**Decision Tree** (Pohon Keputusan) adalah algoritma *Machine Learning* yang cara kerjanya sangat mirip dengan bagan alur (*flowchart*) atau permainan "Tebak Siapa". Model ini memecah data berdasarkan serangkaian pertanyaan "Ya/Tidak" (kondisi *If-Else*) sampai mencapai suatu kesimpulan atau kelompok tertentu.

Bayangkan kamu sedang memutuskan apakah akan membawa payung hari ini:
* *Apakah di luar mendung?* (Jika Ya -> Bawa Payung. Jika Tidak -> Cek hal lain).
* *Apakah ramalan cuaca bilang akan hujan?* ... dan seterusnya.

## 2. Mengapa Menggunakan Decision Tree?
Dalam proyek ini, kita tidak hanya mencari algoritma dengan tingkat tebakan (akurasi) tertinggi. Kita menggunakan Decision Tree karena model ini bersifat **White-Box** (transparan). 

Algoritma lain mungkin pintar menebak, tapi tidak bisa menjelaskan *kenapa* tebakannya begitu. Dengan Decision Tree, kita bisa mengekstrak **Aturan Bisnis (Business Rules)**. Aturan ini sangat vital untuk diserahkan kepada tim *Marketing* agar mereka tahu persis batasan nilai (seperti jumlah hari belanja, total pengeluaran, dll) dari masing-masing tipe pelanggan hasil *clustering* K-Means sebelumnya.

## 3. Alur Kerja
* **Input:** Dataset `hasildata_kmeans-standard.csv` (11 Fitur Numerik kebiasaan pelanggan dan 1 Target Cluster).
* **Output:** *Classification Report* (Rapor performa model) dan Teks Aturan Bisnis (Logika *If-Else*).

In [1]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: Standard K-Means)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-standard.csv')

# Pisahkan Fitur (Var1 sampai Var11) dan Target (Cluster)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: Pisahkan 80% untuk bahan belajar (Train) dan 20% untuk ujian (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Proses Belajar (Training) Model
Sekarang kita menyuruh model untuk mempelajari pola dari data *training*. 

**Catatan Penting:** Kita mengatur `max_depth=4`. Artinya, kita membatasi model agar maksimal hanya boleh bertanya sebanyak 4 tingkat/cabang ke bawah. Kenapa? Agar aturan bisnis yang dihasilkan nanti **tidak terlalu panjang dan rumit** saat dibaca oleh manusia (tim bisnis). Jika dibiarkan tanpa batas, model akan membuat aturan yang sangat detail tapi tidak bisa dipraktikkan di lapangan.

In [2]:
# 2. TRAINING MODEL DECISION TREE
model_dt = DecisionTreeClassifier(random_state=42, max_depth=4)
model_dt.fit(X_train, y_train)

# Minta model menebak data ujian (Test)
prediksi_dt = model_dt.predict(X_test)

# Tampilkan Rapor Performa (Classification Report)
print("=== CLASSIFICATION REPORT: DECISION TREE (STANDARD K-MEANS) ===\n")
print(classification_report(y_test, prediksi_dt))

=== CLASSIFICATION REPORT: DECISION TREE (STANDARD K-MEANS) ===

              precision    recall  f1-score   support

           0       0.88      0.96      0.92       137
           1       0.70      0.87      0.77       211
           2       0.85      0.64      0.73       121
           3       0.95      0.90      0.92       166
           4       0.84      0.76      0.80       149
           5       0.95      0.83      0.88        83

    accuracy                           0.83       867
   macro avg       0.86      0.83      0.84       867
weighted avg       0.84      0.83      0.83       867



### Cara Membaca Classification Report:
* **Accuracy (0.83):** Secara keseluruhan, tebakan model benar **83%** dari total data ujian. Angka ini sudah cukup baik mengingat kita membatasi kedalaman pohonnnya (`max_depth=4`).
* **Precision:** Seberapa sering tebakan model benar. (Misal: Kalau model bilang ini pelanggan kelas 0, seberapa yakin kita itu benar-benar kelas 0?).
* **Recall:** Dari semua pelanggan kelas 0 yang asli, berapa banyak yang berhasil ditemukan oleh model?

## 5. Ekstraksi Aturan Bisnis
Inilah tujuan utama kita. Kita akan menerjemahkan pohon yang sudah dibuat model menjadi teks biasa yang bisa dibaca.

In [3]:
# 3. EKSTRAKSI ATURAN BISNIS
aturan_bisnis = export_text(model_dt, feature_names=fitur)
print("=== BUSINESS RULES (ATURAN LOGIKA PELANGGAN) ===\n")
print(aturan_bisnis)

=== BUSINESS RULES (ATURAN LOGIKA PELANGGAN) ===

|--- Var7 <= 7.35
|   |--- Var1 <= 105.50
|   |   |--- Var9 <= 0.50
|   |   |   |--- Var11 <= 152.01
|   |   |   |   |--- class: 0
|   |   |   |--- Var11 >  152.01
|   |   |   |   |--- class: 5
|   |   |--- Var9 >  0.50
|   |   |   |--- Var4 <= 2195.45
|   |   |   |   |--- class: 0
|   |   |   |--- Var4 >  2195.45
|   |   |   |   |--- class: 2
|   |--- Var1 >  105.50
|   |   |--- Var9 <= 0.50
|   |   |   |--- Var11 <= 232.38
|   |   |   |   |--- class: 3
|   |   |   |--- Var11 >  232.38
|   |   |   |   |--- class: 5
|   |   |--- Var9 >  0.50
|   |   |   |--- Var8 <= 129.50
|   |   |   |   |--- class: 3
|   |   |   |--- Var8 >  129.50
|   |   |   |   |--- class: 3
|--- Var7 >  7.35
|   |--- Var4 <= 721.79
|   |   |--- Var8 <= 96.17
|   |   |   |--- Var7 <= 46.38
|   |   |   |   |--- class: 0
|   |   |   |--- Var7 >  46.38
|   |   |   |   |--- class: 4
|   |   |--- Var8 >  96.17
|   |   |   |--- Var9 <= 0.50
|   |   |   |   |--- class: 5


### Cara Membaca Aturan Bisnis di Atas:
Bagan di atas adalah panduan bagi *Marketing* untuk mengklasifikasikan pelanggan baru. Cara membacanya dari atas ke bawah. 

Sebagai contoh, coba lihat baris pertama hingga menemukan `class: 0`:
1. Apakah pelanggan memiliki nilai **Var7 <= 7.35**? 
2. Jika Ya, cek lagi: Apakah **Var1 <= 105.50**?
3. Jika Ya, cek lagi: Apakah **Var9 <= 0.50**?
4. Jika Ya, cek lagi: Apakah **Var11 <= 152.01**?
5. Jika Ya, maka dipastikan pelanggan ini masuk ke **Cluster 0**.

*Catatan: Tim bisnis nantinya tinggal mengganti teks `Var1`, `Var7`, dst., dengan nama asli kebiasaan pelanggannya (misal: `Total Belanja`, `Jumlah Kunjungan`, dll).*

## 6. Menyimpan Model
Agar model ini bisa dipakai di aplikasi atau dipanggil di masa depan tanpa harus dilatih ulang dari awal, kita akan menyimpannya ke dalam format `.pkl`.

In [4]:
# 4. EXPORT MODEL KE FOLDER 'models'
# Buat foldernya dulu jika belum ada
os.makedirs('../models', exist_ok=True)

# Simpan model
joblib.dump(model_dt, '../models/model_dt_classification_standard.pkl')
print("\n[SUCCESS] Model Decision Tree berhasil diekspor ke '../models/model_dt_classification_standard.pkl'")


[SUCCESS] Model Decision Tree berhasil diekspor ke '../models/model_dt_classification_standard.pkl'
